# Transformer-Based GNN Pipeline for Disease Link Prediction
**What this does:**
1. Generates synthetic EHR data (2,000 patients, 20 diseases)
2. Builds a disease co-occurrence graph with 4 node features
3. Manually splits edges 80% train / 20% test (no data leakage)
4. Trains 4 GNN architectures: GCN, GAT, GraphSAGE, Graph Transformer
5. Evaluates on held-out test edges with ROC-AUC, PR curves, confusion matrix
6. FAISS similarity search on node embeddings

> **Enable GPU**: Runtime → Change runtime type → T4 GPU

In [ ]:
# ── Cell 1: Install dependencies ──
import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
!pip install -q torch-geometric
!pip install -q faiss-cpu scikit-learn matplotlib pandas numpy

In [ ]:
# ── Cell 2: Imports ──
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import faiss
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, TransformerConv
from torch_geometric.utils import negative_sampling
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ── Cell 3: Generate synthetic EHR data (50 diseases, 10,000 patients) ──
DISEASES = [
    # Metabolic
    'Type 2 Diabetes', 'Obesity', 'Hyperlipidemia', 'Hypothyroidism', 'Gout', 'NAFLD',
    # Cardiovascular
    'Hypertension', 'Heart Failure', 'Coronary Artery Disease', 'Atrial Fibrillation',
    'Peripheral Artery Disease', 'Stroke', 'Aortic Stenosis', 'Deep Vein Thrombosis',
    # Pulmonary
    'COPD', 'Asthma', 'Sleep Apnea', 'Pulmonary Hypertension', 'Pulmonary Embolism',
    # Renal & GI
    'Chronic Kidney Disease', 'Kidney Stones', 'GERD', 'IBD', 'IBS', 'Cirrhosis',
    # Musculoskeletal
    'Osteoarthritis', 'Osteoporosis', 'Rheumatoid Arthritis', 'Fibromyalgia', 'Gout Arthritis',
    # Neurological
    'Depression', 'Anxiety Disorder', 'Bipolar Disorder', 'PTSD', 'Dementia',
    "Parkinson's Disease", 'Epilepsy', 'Migraines', 'ADHD',
    # Oncology
    'Anemia', 'Cancer (unspecified)', 'Prostate Cancer', 'Breast Cancer', 'Colon Cancer',
    # Skin & Immune
    'Psoriasis', 'Eczema', 'Lupus', 'Multiple Sclerosis',
    # Other
    'Hypothyroidism (Auto)', 'Chronic Pain', 'Vitamin D Deficiency', 'Fatty Liver'
]
N = len(DISEASES)
print(f'Total diseases: {N}')

COMORBIDITY_PAIRS = {
    ('Type 2 Diabetes', 'Hypertension'): 0.75,
    ('Type 2 Diabetes', 'Obesity'): 0.72,
    ('Type 2 Diabetes', 'Hyperlipidemia'): 0.65,
    ('Type 2 Diabetes', 'Chronic Kidney Disease'): 0.42,
    ('Type 2 Diabetes', 'NAFLD'): 0.55,
    ('Type 2 Diabetes', 'Coronary Artery Disease'): 0.48,
    ('Obesity', 'Sleep Apnea'): 0.60,
    ('Obesity', 'NAFLD'): 0.65,
    ('Obesity', 'Hypertension'): 0.65,
    ('Obesity', 'Osteoarthritis'): 0.45,
    ('Heart Failure', 'Atrial Fibrillation'): 0.62,
    ('Heart Failure', 'Hypertension'): 0.70,
    ('Heart Failure', 'Coronary Artery Disease'): 0.68,
    ('Heart Failure', 'Chronic Kidney Disease'): 0.50,
    ('COPD', 'Asthma'): 0.35,
    ('COPD', 'Heart Failure'): 0.28,
    ('COPD', 'Pulmonary Hypertension'): 0.30,
    ('Depression', 'Anxiety Disorder'): 0.68,
    ('Depression', 'PTSD'): 0.52,
    ('Depression', 'Chronic Pain'): 0.55,
    ('Hypertension', 'Coronary Artery Disease'): 0.55,
    ('Hypertension', 'Stroke'): 0.50,
    ('Stroke', 'Atrial Fibrillation'): 0.48,
    ('Osteoporosis', 'Rheumatoid Arthritis'): 0.42,
    ('Psoriasis', 'Rheumatoid Arthritis'): 0.38,
    ('Anemia', 'Chronic Kidney Disease'): 0.52,
    ('Epilepsy', 'Depression'): 0.40,
    ('Migraines', 'Anxiety Disorder'): 0.45,
    ('IBD', 'Anemia'): 0.48,
    ('Fatty Liver', 'NAFLD'): 0.70,
}

def generate_patient_records(n_patients=10000, seed=42):
    np.random.seed(seed)
    records = []
    for pid in range(n_patients):
        n_d = np.random.choice([1,2,3,4,5], p=[0.20,0.30,0.28,0.15,0.07])
        base = list(np.random.choice(DISEASES, size=min(n_d, N), replace=False))
        extra = []
        for d in base:
            for (d1, d2), prob in COMORBIDITY_PAIRS.items():
                if d == d1 and d2 not in base and np.random.random() < prob * 0.5:
                    extra.append(d2)
                elif d == d2 and d1 not in base and np.random.random() < prob * 0.5:
                    extra.append(d1)
        records.append({'patient_id': f'P{pid:05d}', 'diseases': list(set(base+extra))})
    return records

def build_cooccurrence_matrix(records):
    d2i = {d: i for i, d in enumerate(DISEASES)}
    matrix = np.zeros((N, N), dtype=int)
    prevalence = np.zeros(N, dtype=int)
    for r in records:
        ds = [d for d in r['diseases'] if d in d2i]
        for d in ds: prevalence[d2i[d]] += 1
        for i, d1 in enumerate(ds):
            for d2 in ds[i+1:]:
                i1, i2 = d2i[d1], d2i[d2]
                matrix[i1][i2] += 1; matrix[i2][i1] += 1
    return matrix, prevalence

print('Generating 10,000 patient records...')
records = generate_patient_records(10000)
matrix, prevalence = build_cooccurrence_matrix(records)
print(f'Done. Co-occurrence matrix: {matrix.shape}')
print(f'Max co-occurrence: {matrix.max()} | Mean (non-zero): {matrix[matrix>0].mean():.1f}')

In [ ]:
# ── Cell 4: Build graph + hard 80/20 edge split ──
# Key fix 1: one-hot node features (no degree/connectivity leakage)
# Key fix 2: hard negatives = mid-range co-occurrence pairs (not zero)
#            Forces model to distinguish strong vs medium comorbidity

# One-hot: each disease gets its own unique feature dimension
x = torch.eye(N, dtype=torch.float).to(device)
print(f'Node feature shape: {x.shape}  (one-hot, no leakage)')

# Compute percentile thresholds from the full co-occurrence distribution
flat = sorted([matrix[i][j] for i in range(N) for j in range(i+1, N)])
n_pairs = len(flat)
p75 = flat[int(0.75 * n_pairs)]   # top 25% = strong comorbidity (positives)
p40 = flat[int(0.40 * n_pairs)]   # 40th–60th percentile = hard negatives
p60 = flat[int(0.60 * n_pairs)]
print(f'Co-occurrence thresholds — pos (>{p75}): strong | neg ({p40}–{p60}): hard negatives')

# Positive edges: strong comorbidity (top 25%)
all_pos = [(min(i,j), max(i,j)) for i in range(N) for j in range(i+1,N)
           if matrix[i][j] >= p75]

# Negative edges: medium co-occurrence (hard negatives, 40th–60th pct)
all_neg = [(min(i,j), max(i,j)) for i in range(N) for j in range(i+1,N)
           if p40 <= matrix[i][j] < p60]

print(f'Positive edges: {len(all_pos)} | Negative edges (hard): {len(all_neg)}')

# 80/20 split — balance test set
train_pos, test_pos = train_test_split(all_pos, test_size=0.2, random_state=42)
train_neg, test_neg = train_test_split(all_neg, test_size=0.2, random_state=42)
min_test = min(len(test_pos), len(test_neg))
test_pos, test_neg = test_pos[:min_test], test_neg[:min_test]
print(f'Train: {len(train_pos)} pos / {len(train_neg)} neg')
print(f'Test:  {len(test_pos)} pos / {len(test_neg)} neg (balanced)')

# Build edge tensors
def to_bidir(pairs):
    arr = np.array(pairs)
    return torch.tensor(np.concatenate([arr, arr[:,::-1]], axis=0).T, dtype=torch.long).to(device)

train_ei     = to_bidir(train_pos)
train_neg_ei = to_bidir(train_neg)
test_pos_ei  = torch.tensor(np.array(test_pos).T, dtype=torch.long).to(device)
test_neg_ei  = torch.tensor(np.array(test_neg).T, dtype=torch.long).to(device)

print(f'\nGraph: {N} nodes | {train_ei.size(1)//2} train edges | {x.shape[1]}-dim features')

In [ ]:
# ── Cell 5: Define GNN models ──
class GCN(torch.nn.Module):
    def __init__(self, in_ch, h, out_ch):
        super().__init__()
        self.conv1 = GCNConv(in_ch, h)
        self.conv2 = GCNConv(h, out_ch)
    def encode(self, x, ei):
        return self.conv2(F.dropout(F.relu(self.conv1(x, ei)), p=0.3, training=self.training), ei)
    def decode(self, z, ei): return (z[ei[0]] * z[ei[1]]).sum(dim=-1)
    def forward(self, x, ei): return self.encode(x, ei)

class GAT(torch.nn.Module):
    def __init__(self, in_ch, h, out_ch, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_ch, h, heads=heads, dropout=0.3)
        self.conv2 = GATConv(h*heads, out_ch, heads=1, concat=False, dropout=0.3)
    def encode(self, x, ei):
        x = F.elu(self.conv1(F.dropout(x, p=0.3, training=self.training), ei))
        return self.conv2(F.dropout(x, p=0.3, training=self.training), ei)
    def decode(self, z, ei): return (z[ei[0]] * z[ei[1]]).sum(dim=-1)
    def forward(self, x, ei): return self.encode(x, ei)

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_ch, h, out_ch):
        super().__init__()
        self.conv1 = SAGEConv(in_ch, h)
        self.conv2 = SAGEConv(h, out_ch)
    def encode(self, x, ei):
        return self.conv2(F.dropout(F.relu(self.conv1(x, ei)), p=0.3, training=self.training), ei)
    def decode(self, z, ei): return (z[ei[0]] * z[ei[1]]).sum(dim=-1)
    def forward(self, x, ei): return self.encode(x, ei)

class GraphTransformer(torch.nn.Module):
    def __init__(self, in_ch, h, out_ch, heads=4):
        super().__init__()
        self.conv1 = TransformerConv(in_ch, h, heads=heads, dropout=0.3)
        self.conv2 = TransformerConv(h*heads, out_ch, heads=1, concat=False, dropout=0.3)
    def encode(self, x, ei):
        return self.conv2(F.dropout(F.relu(self.conv1(x, ei)), p=0.3, training=self.training), ei)
    def decode(self, z, ei): return (z[ei[0]] * z[ei[1]]).sum(dim=-1)
    def forward(self, x, ei): return self.encode(x, ei)

print('Models defined: GCN, GAT, GraphSAGE, GraphTransformer')

In [ ]:
# ── Cell 6: Train on train edges only ──
def train_model(model, x, train_ei, train_neg_ei, epochs=300, lr=0.005):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
    for epoch in range(1, epochs+1):
        model.train()
        optimizer.zero_grad()
        z = model(x, train_ei)
        pos_s = model.decode(z, train_ei)
        neg_s = model.decode(z, train_neg_ei)
        # Balance pos/neg counts
        min_count = min(pos_s.size(0), neg_s.size(0))
        scores = torch.cat([pos_s[:min_count], neg_s[:min_count]])
        labels = torch.cat([
            torch.ones(min_count),
            torch.zeros(min_count)
        ]).to(device)
        loss = F.binary_cross_entropy_with_logits(scores, labels)
        loss.backward(); optimizer.step(); scheduler.step()
        if epoch % 75 == 0:
            print(f'  Epoch {epoch:3d} | Loss: {loss.item():.4f}')
    return model

IN = x.size(1)  # 4 features
models = {
    'GCN':               GCN(IN, 64, 32),
    'GAT':               GAT(IN, 64, 32),
    'GraphSAGE':         GraphSAGE(IN, 64, 32),
    'Graph Transformer': GraphTransformer(IN, 64, 32),
}

for name, model in models.items():
    print(f'\n--- Training {name} ---')
    models[name] = train_model(model, x, train_ei, train_neg_ei)

In [ ]:
# ── Cell 7: Evaluate on held-out test edges ──
@torch.no_grad()
def evaluate(model, x, train_ei, test_pos_ei, test_neg_ei):
    model.eval()
    z = model(x, train_ei)  # encode using train graph
    pos_s = torch.sigmoid(model.decode(z, test_pos_ei)).cpu().numpy()
    neg_s = torch.sigmoid(model.decode(z, test_neg_ei)).cpu().numpy()
    scores = np.concatenate([pos_s, neg_s])
    labels = np.concatenate([np.ones(len(pos_s)), np.zeros(len(neg_s))])
    return scores, labels, z.cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].set_title('ROC Curves — Test Set')
axes[0].plot([0,1],[0,1],'k--',label='Random')
axes[1].set_title('Precision-Recall Curves — Test Set')

results = {}
for name, model in models.items():
    scores, labels, embeddings = evaluate(model, x, train_ei, test_pos_ei, test_neg_ei)
    auc = roc_auc_score(labels, scores)
    ap  = average_precision_score(labels, scores)
    fpr, tpr, _ = roc_curve(labels, scores)
    prec, rec, _ = precision_recall_curve(labels, scores)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
    axes[1].plot(rec, prec, label=f'{name} (AP={ap:.3f})')
    results[name] = {'ROC-AUC': auc, 'AP': ap, 'embeddings': embeddings}
    print(f'{name:20s} | ROC-AUC: {auc:.4f} | Avg Precision: {ap:.4f}')

best = max(results, key=lambda k: results[k]['ROC-AUC'])
best_scores, best_labels, _ = evaluate(models[best], x, train_ei, test_pos_ei, test_neg_ei)
cm = confusion_matrix(best_labels, (best_scores > 0.5).astype(int))
ConfusionMatrixDisplay(cm, display_labels=['No Link','Link']).plot(ax=axes[2])
axes[2].set_title(f'Confusion Matrix — {best}')

axes[0].legend(); axes[1].legend()
plt.tight_layout()
plt.savefig('results_comparison.png', dpi=150)
plt.show()
print(f'\nBest model: {best} (ROC-AUC: {results[best]["ROC-AUC"]:.4f})')

In [ ]:
# ── Cell 8: FAISS similarity search ──
def faiss_similar_diseases(embeddings, disease_names, query_disease, top_k=5):
    emb = embeddings.astype(np.float32)
    faiss.normalize_L2(emb)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    q_idx = disease_names.index(query_disease)
    distances, indices = index.search(emb[q_idx:q_idx+1], top_k+1)
    print(f"\nDiseases most similar to '{query_disease}':")
    for idx, score in zip(indices[0], distances[0]):
        if disease_names[idx] != query_disease:
            print(f'  {disease_names[idx]:35s} similarity: {score:.4f}')

best_embeddings = results[best]['embeddings']
faiss_similar_diseases(best_embeddings, DISEASES, 'Type 2 Diabetes')
faiss_similar_diseases(best_embeddings, DISEASES, 'Heart Failure')